# Phase 2 — REA Environmental Exclusions
**Ontario Distribution-Connected Solar Siting | 10+ MW Projects**

Build the Renewable Energy Approval (REA) exclusion layer from nine environmental
constraint tables (each buffered by its REA-mandated setback distance), then subtract
it from the Phase 1 selected lots to produce net-developable geometry and area.

**Input**
- `analysis.selected_lots_{county_slug}` — Phase 1 Step 3 output (lots intersecting OEM ≥ 10 MW)
- `analysis.aoi_county_{county_slug}` — county AOI for spatial filtering of constraint layers

**Steps**
1. Build REA exclusion layer (clip, buffer, dissolve, repair)
2. Subtract exclusions from lots → net-developable geometry + area
3. Verification & interactive map

---
Change only `COUNTY_NAME` in the Configuration cell to reproduce for any county.

## 0 · Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PROJECT PARAMETERS — edit this cell only to run for a different county
# ─────────────────────────────────────────────────────────────────────────────

import os

COUNTY_NAME    = "Peterborough"          # Must match adm_district_township.name_2
AOI_BUFFER_M   = 2000             # AOI buffer in metres (EPSG:5321)
MIN_POLYGON_HA = 30               # OEM polygon noise filter (hectares)
MIN_NET_ACRES  = 40               # Minimum net acres after REA exclusions
CONSERVATION_AUTH = "orca"  # Conservation authority to use for REA exclusions — must match ca.name

# PostGIS connection
PG_CONN = os.environ["POSTGRES_CONNECTION_STRING"]

# Output schema — tables are named per step and county automatically
OUTPUT_SCHEMA = "analysis"

# CRS
CRS_WGS84   = 4326
CRS_NAD83   = 4269   # NAD83 geographic — bridge CRS
CRS_ONTARIO = 5321   # NAD83(CSRS) / Ontario MNR Lambert — all calculations

## 1 · Environment Setup & Utilities

In [2]:
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import MiniMap
from branca.colormap import linear
from sqlalchemy import create_engine, text

engine = create_engine(PG_CONN)

def read_postgis(sql: str, geom_col: str = "geom") -> gpd.GeoDataFrame:
    """Execute a PostGIS query and return a WGS84 GeoDataFrame."""
    gdf = gpd.read_postgis(sql, engine, geom_col=geom_col)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=CRS_WGS84)
    return gdf.to_crs(epsg=CRS_WGS84)


def save_to_postgis(gdf: gpd.GeoDataFrame, table: str, label: str) -> None:
    """Write a GeoDataFrame to PostGIS in EPSG:5321 and create a GiST spatial index."""
    geom_col = gdf.geometry.name
    gdf.to_crs(epsg=CRS_ONTARIO).to_postgis(
        name=table,
        con=engine,
        schema=OUTPUT_SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=500
    )
    with engine.connect() as conn:
        conn.execute(text(
            f"CREATE INDEX IF NOT EXISTS {table}_geom_idx "
            f"ON {OUTPUT_SCHEMA}.{table} USING GIST({geom_col})"
        ))
        conn.commit()
    print(f"  {label:35s} → {OUTPUT_SCHEMA}.{table} "
          f"({len(gdf):,} rows, EPSG:{CRS_ONTARIO}, GiST index)")


county_slug = COUNTY_NAME.lower().replace(" ", "_")

with engine.connect() as conn:
    print("PostGIS:", conn.execute(text("SELECT PostGIS_Full_Version()")).scalar())
print(f"County : {COUNTY_NAME} (slug: {county_slug})")

PostGIS: POSTGIS="3.3.4 0" [EXTENSION] PGSQL="150" GEOS="3.11.2-CAPI-1.17.2" SFCGAL="SFCGAL 1.4.1, CGAL 5.5.2, BOOST 1.82.0" PROJ="9.2.1" GDAL="GDAL 3.6.4, released 2023/04/17" LIBXML="2.11.4" LIBJSON="0.16" LIBPROTOBUF="1.4.1" WAGYU="0.5.0 (Internal)" TOPOLOGY RASTER
County : Peterborough (slug: peterborough)


---
## Step 1 — Build REA Exclusion Layer

Constructs a single unified exclusion geometry from all REA environmental constraint layers
clipped to the county AOI. Nine layers from the `rea` schema:

| Layer | Buffer | Notes |
|---|---|---|
| `ansi` | `buffer_dist` | Areas of Natural & Scientific Interest |
| `conservation_reserve` | `buffer_dist` | Conservation reserves |
| `orca_regulated_area` | `buffer_dist` | ORCA (Conservation Authority) regulated areas |
| `prov_park_zone` | `buffer_dist` | Provincial park zones |
| `waterbody` | `buffer_dist` | Lakes, ponds, reservoirs |
| `watercourse` | `buffer_dist` | Rivers, streams, creeks |
| `wetland` | `buffer_dist` | Provincially significant wetlands |
| `woodland` | `buffer_dist` | Significant woodlands |
| `disturbance_exclusions` | *(none)* | Hard exclusion — clip only |

**Pipeline**: clip to AOI → buffer by `buffer_dist` → UNION ALL → dissolve →
repair (`ST_MakeValid` → `ST_Buffer(0)` → `ST_SimplifyPreserveTopology(1)` →
`ST_SnapToGrid(0.01)` → final `ST_MakeValid`).

In [3]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Layers that use buffer_dist
BUFFERED_LAYERS = [
    "ansi",
    "conservation_reserve",
    f"{CONSERVATION_AUTH}_regulated_area",
    "prov_park_zone",
    "waterbody",
    "watercourse",
    "wetland",
    "woodland",
]

# Build one CTE per buffered layer
buf_ctes = []
for layer in BUFFERED_LAYERS:
    cte_name = f"{layer}_buf"
    buf_ctes.append(f"""
{cte_name} AS (
    SELECT
        ST_Buffer(
            ST_Intersection(ST_Transform(L.geom, {CRS_ONTARIO}), A.geom),
            L.buffer_dist
        ) AS geom
    FROM rea.{layer} L
    JOIN aoi A ON ST_Intersects(ST_Transform(L.geom, {CRS_ONTARIO}), A.geom)
    WHERE L.buffer_dist IS NOT NULL AND L.buffer_dist > 0
)""")

# Disturbance exclusions — clip only, no buffer
dist_cte = f"""
disturbance_exclusions_clip AS (
    SELECT
        ST_Intersection(ST_Transform(L.geom, {CRS_ONTARIO}), A.geom) AS geom
    FROM rea.disturbance_exclusions L
    JOIN aoi A ON ST_Intersects(ST_Transform(L.geom, {CRS_ONTARIO}), A.geom)
)"""

# UNION ALL all CTEs
all_cte_names = [f"{layer}_buf" for layer in BUFFERED_LAYERS] + ["disturbance_exclusions_clip"]
merge_parts = "\nUNION ALL\n".join(
    f"SELECT geom FROM {name}" for name in all_cte_names
)

sql = f"""
WITH
aoi AS (
    SELECT ST_Union(geom) AS geom
    FROM {OUTPUT_SCHEMA}.aoi_county_{county_slug}
),
{','.join(buf_ctes)},
{dist_cte},
merged AS (
    {merge_parts}
),
dissolved AS (
    SELECT ST_Union(geom) AS geom
    FROM merged
),
repaired AS (
    SELECT
        ST_MakeValid(
            ST_SnapToGrid(
                ST_SimplifyPreserveTopology(
                    ST_Buffer(ST_MakeValid(geom), 0),
                    1       -- 1 m tolerance (EPSG:5321)
                ),
                0.01    -- snap to 1 cm grid
            )
        ) AS geom
    FROM dissolved
)
SELECT
    ST_Transform(geom, {CRS_WGS84}) AS geom
FROM repaired
"""

gdf_rea = read_postgis(sql)

print(f"REA exclusion layer for '{COUNTY_NAME}':")
print(f"  Geometry type : {gdf_rea.geometry.geom_type.iloc[0]}")
print(f"  Area          : {gdf_rea.to_crs(epsg=CRS_ONTARIO).geometry.area.sum() / 10000:,.1f} ha")

save_to_postgis(gdf_rea, f"rea_exclusions_{county_slug}", "Step 1 — REA exclusion layer")

REA exclusion layer for 'Peterborough':
  Geometry type : MultiPolygon
  Area          : 419,254.2 ha
  Step 1 — REA exclusion layer        → analysis.rea_exclusions_peterborough (1 rows, EPSG:5321, GiST index)


---
## Step 2 — Subtract REA Exclusions from Selected Lots

Uses `ST_Difference` to erase the REA exclusion geometry from each selected lot,
producing net-developable geometry. Computes:
- **`acre_net`** — net developable area after exclusions (acres)
- **`pct_remaining`** — percentage of original lot area surviving

Lots with net area below `MIN_NET_ACRES` are filtered out.
`LEFT JOIN` + `COALESCE` preserves lots with no exclusion overlap.

In [4]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_lots_rea = read_postgis(f"""
WITH exclusions AS (
    SELECT ST_Union(geom) AS geom
    FROM {OUTPUT_SCHEMA}.rea_exclusions_{county_slug}
)
SELECT
    l.lot_ident,
    l.concession_ident,
    l.geographic_township_name,
    l.acre_total,
    l.grid_capacity_mw,
    l.voltage_3ph,
    l.station_name,
    l.feeder_name,
    l.ldc_name,
    ROUND((ST_Area(ST_Difference(
        ST_Transform(l.geom, {CRS_ONTARIO}),
        COALESCE(e.geom, ST_GeomFromText('GEOMETRYCOLLECTION EMPTY', {CRS_ONTARIO}))
    )) / 4046.856)::numeric, 2) AS acre_net,
    ROUND((ST_Area(ST_Difference(
        ST_Transform(l.geom, {CRS_ONTARIO}),
        COALESCE(e.geom, ST_GeomFromText('GEOMETRYCOLLECTION EMPTY', {CRS_ONTARIO}))
    )) / ST_Area(ST_Transform(l.geom, {CRS_ONTARIO})) * 100)::numeric, 1) AS pct_remaining,
    ST_Transform(
        ST_MakeValid(ST_Difference(
            ST_Transform(l.geom, {CRS_ONTARIO}),
            COALESCE(e.geom, ST_GeomFromText('GEOMETRYCOLLECTION EMPTY', {CRS_ONTARIO}))
        )),
        {CRS_WGS84}
    ) AS geom
FROM {OUTPUT_SCHEMA}.selected_lots_{county_slug} l
LEFT JOIN exclusions e ON ST_Intersects(ST_Transform(l.geom, {CRS_ONTARIO}), e.geom)
WHERE ST_Area(ST_Difference(
        ST_Transform(l.geom, {CRS_ONTARIO}),
        COALESCE(e.geom, ST_GeomFromText('GEOMETRYCOLLECTION EMPTY', {CRS_ONTARIO}))
      )) / 4046.856 > {MIN_NET_ACRES}
ORDER BY acre_net DESC
""")

print(f"Lots after REA exclusions: {len(gdf_lots_rea):,}")
print(f"Net area range: {gdf_lots_rea['acre_net'].min():.1f} – {gdf_lots_rea['acre_net'].max():.1f} ac")
print(f"Mean pct remaining: {gdf_lots_rea['pct_remaining'].mean():.1f}%")

save_to_postgis(gdf_lots_rea, f"lots_developable_{county_slug}", "Step 2 — Lots after REA exclusions")

Lots after REA exclusions: 404
Net area range: 40.3 – 197.8 ac
Mean pct remaining: 45.9%
  Step 2 — Lots after REA exclusions  → analysis.lots_developable_peterborough (404 rows, EPSG:5321, GiST index)


---
## Step 3 — Verification & Map

Summary statistics and an interactive Folium map:
- **Lots** colored by `pct_remaining` (green = mostly intact, red = heavily excluded)
- **REA exclusion zones** — semi-transparent red overlay
- **OEM grid zones** — light context layer
- Tooltips with lot attributes + net area

In [5]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# ── Summary statistics ─────────────────────────────────────────────────────
gdf_phase1_lots = read_postgis(f"""
    SELECT lot_ident, acre_total, geom
    FROM {OUTPUT_SCHEMA}.selected_lots_{county_slug}
""")
n_before = len(gdf_phase1_lots)
n_after  = len(gdf_lots_rea)
n_eliminated = n_before - n_after

print(f"Phase 2 — REA Exclusion Summary for '{COUNTY_NAME}'")
print(f"{'─' * 50}")
print(f"  Lots before exclusions : {n_before:,}")
print(f"  Lots after exclusions  : {n_after:,}")
print(f"  Lots eliminated        : {n_eliminated:,} (below {MIN_NET_ACRES} net acres)")
print(f"  Total net acreage      : {gdf_lots_rea['acre_net'].sum():,.1f} ac")
print(f"  Mean pct remaining     : {gdf_lots_rea['pct_remaining'].mean():.1f}%")
print(f"  Median pct remaining   : {gdf_lots_rea['pct_remaining'].median():.1f}%")

Phase 2 — REA Exclusion Summary for 'Peterborough'
──────────────────────────────────────────────────
  Lots before exclusions : 1,469
  Lots after exclusions  : 404
  Lots eliminated        : 1,065 (below 40 net acres)
  Total net acreage      : 37,251.2 ac
  Mean pct remaining     : 45.9%
  Median pct remaining   : 44.6%


In [6]:
# ── Interactive map ─────────────────────────────────────────────────────────
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Load context layers
gdf_oem = read_postgis(f"""
    SELECT station_name, feeder_name, capacity, geom
    FROM {OUTPUT_SCHEMA}.oem_county_{county_slug}
""")

# Map center from lots
center = [
    gdf_lots_rea.geometry.centroid.y.mean(),
    gdf_lots_rea.geometry.centroid.x.mean()
]

# Colormaps — pct_remaining: green (100%) → red (low)
pct_vals = gdf_lots_rea["pct_remaining"].dropna()
pct_cmap = linear.RdYlGn_09.scale(pct_vals.min(), pct_vals.max())
pct_cmap.caption = "% Lot Remaining After REA Exclusions"

m = folium.Map(location=center, zoom_start=10, tiles="CartoDB positron")
MiniMap(toggle_display=True).add_to(m)

# Layer 1 — OEM grid (context)
folium.GeoJson(
    gdf_oem.__geo_interface__,
    name="OEM grid (>10 MW)",
    style_function=lambda _: {
        "fillColor": "#3498db", "color": "#2471a3",
        "weight": 0.6, "fillOpacity": 0.2
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["station_name", "feeder_name", "capacity"],
        aliases=["Station", "Feeder", "Capacity (MW)"],
        localize=True
    )
).add_to(m)

# Layer 2 — REA exclusion zones
folium.GeoJson(
    gdf_rea.__geo_interface__,
    name="REA exclusion zones",
    style_function=lambda _: {
        "fillColor": "#e74c3c", "color": "#c0392b",
        "weight": 0.5, "fillOpacity": 0.35
    }
).add_to(m)

# Layer 3 — Lots colored by pct_remaining
folium.GeoJson(
    gdf_lots_rea.__geo_interface__,
    name="Developable lots (after REA)",
    style_function=lambda feat: {
        "fillColor": pct_cmap(feat["properties"].get("pct_remaining") or 0),
        "color": "#2c3e50", "weight": 0.8, "fillOpacity": 0.75
    },
    highlight_function=lambda _: {
        "color": "#1a3c2d", "weight": 2.5, "fillOpacity": 0.9
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["lot_ident", "concession_ident", "geographic_township_name",
                "acre_total", "acre_net", "pct_remaining",
                "grid_capacity_mw", "station_name"],
        aliases=["Lot", "Concession", "Township",
                 "Area (ac)", "Net Area (ac)", "% Remaining",
                 "Grid (MW)", "Station"],
        localize=True, sticky=True
    )
).add_to(m)

pct_cmap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

out_html = f"map_phase2_{county_slug}.html"
m.save(out_html)
print(f"Saved: {out_html}")

Saved: map_phase2_peterborough.html


---
## Verification — Phase 2 Tables

In [7]:
# Results already saved to analysis schema after each step.
county_slug = COUNTY_NAME.lower().replace(" ", "_")
print("Phase 2 tables in analysis schema:")
with engine.connect() as conn:
    rows = conn.execute(text(f"""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = '{OUTPUT_SCHEMA}'
          AND table_name LIKE '%{county_slug}%'
        ORDER BY table_name
    """)).fetchall()
    for r in rows:
        print(f"  {OUTPUT_SCHEMA}.{r[0]}")

Phase 2 tables in analysis schema:
  analysis.aoi_county_peterborough
  analysis.lots_developable_peterborough
  analysis.oem_county_peterborough
  analysis.rea_exclusions_peterborough
  analysis.selected_lots_peterborough
